# Ranking predictions - adding roster information

The previous notebooks only use the **team's own history** (past points, goals, coach...). That information is
largely exhausted: a team can change a lot between two seasons (stars leaving or arriving, a new goalie) without
its past results showing it. `Scoring.csv` and `Goalies.csv` contain exactly that missing information.

New features (all talent values come from the **previous** season, `roster_features.py`):

| group | features | idea |
|---|---|---|
| `roster_talent` | `roster_prev_pts_z`, `top5_prev_pts_z` | previous-season points of the players on the roster (depth and top line) |
| `talent_moves` | `talent_added_z`, `talent_lost_z`, `net_talent_z` | points brought by newcomers vs points left by departed players |
| `continuity` | `continuity_z` | share of the roster's games played by players already on the team last season |
| `goalie` | `goalie_prev_sv_z` | previous-season save % (shrunk toward league average) of the team's main goalie |

Protocol (same as the other notebooks, no peeking at the test seasons):
1. test whether the roster adds information on seasons **< 2005**,
2. keep only the groups that improve walk-forward validation,
3. evaluate on seasons **>= 2005** against the naive model and the team-only Ridge.

> **Assumption / limitation**: the roster of the season to predict is taken from `Scoring.csv` (players who played
> for the team that year), which also contains players acquired during the season. A strict "day before the season"
> roster is not available in the data, so part of the gain may come from in-season trades (good teams buy players).
> Talent values themselves never use the predicted season.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.ensemble import HistGradientBoostingRegressor, VotingRegressor

import ranking_utils as ru
import roster_features as rf
RS = ru.RANDOM_STATE

## 1. Data: team features + roster features

In [ ]:
df = ru.build_dataset(ru.load_teams())
groups = ru.feature_groups(df)

scoring = pd.read_csv(Path("data") / "Scoring.csv")
goalies = pd.read_csv(Path("data") / "Goalies.csv")
roster = rf.build_roster_features(scoring, goalies)

n_before = len(df)
df = df.merge(roster, on=["year", "tmID"], how="left")
assert len(df) == n_before
ROSTER_COLS = [c for cols in rf.ROSTER_GROUPS.values() for c in cols]
coverage = df[ROSTER_COLS].notna().all(axis=1).mean()
print(f"Roster features available for {coverage:.1%} of team-seasons")
df[ROSTER_COLS] = df[ROSTER_COLS].fillna(0.0)

# Base (team-only) features: the ones selected in ranking_analysis.ipynb
selection_file = Path("ranking_selected_features.json")
BASE = json.loads(selection_file.read_text())["features"] if selection_file.exists() else ru.flatten(groups)
print("Base features:", BASE)

EVAL_YEARS = sorted(int(y) for y in df.loc[df["year"] >= ru.TEST_START_YEAR, "year"].unique())
dev = df[df["year"] < ru.TEST_START_YEAR]
SELECTION_YEARS = [y for y in sorted(dev["year"].unique()) if y >= 1995]
print("Evaluation seasons:", EVAL_YEARS)
display(df.loc[df["year"] == df["year"].max(), ["tmID", "conf_rank"] + ROSTER_COLS].sort_values("conf_rank").head(8).round(2))

## 2. Does the roster add information beyond the team's history?

We regress the season points on last season's points and correlate the **residual** (what the history does not
explain) with each roster feature, on seasons < 2005 only.

In [ ]:
sub = dev[dev["n_hist"] >= 1].copy()
slope, intercept = np.polyfit(sub["pts_z_lag1"], sub["target"], 1)
sub["residual"] = sub["target"] - (intercept + slope * sub["pts_z_lag1"])
corr = pd.Series({c: sub[c].corr(sub["residual"], method="spearman") for c in ROSTER_COLS}).sort_values()
display(corr.round(3).to_frame("spearman with residual"))

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.barh(corr.index, corr.values, color=np.where(corr.values >= 0, "#4C72B0", "#C44E52"))
ax.set_title("Roster signal beyond last season's points"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 3. Which roster groups to keep? (walk-forward on seasons < 2005)

Forward selection of groups on top of the base features with a Ridge model; a group is kept if it lowers the
validation MAE by at least `MIN_GAIN` rank positions.

In [ ]:
MIN_GAIN = 0.01
ridge_sel = lambda: make_pipeline(StandardScaler(), Ridge(alpha=10.0))

def selection_mae(features):
    pred = ru.walk_forward(dev, ridge_sel, features, SELECTION_YEARS)
    return ru.season_metrics(ru.rank_predictions(pred, df))["mae"].mean()

base_mae = selection_mae(BASE)
selected_groups, features_now, best_mae = [], list(BASE), base_mae
steps = [{"added": "(base features)", "mae": base_mae}]
remaining = list(rf.ROSTER_GROUPS)
while remaining:
    trials = {g: selection_mae(features_now + rf.ROSTER_GROUPS[g]) for g in remaining}
    cand = min(trials, key=trials.get)
    print({g: round(v, 3) for g, v in trials.items()})
    if best_mae - trials[cand] < MIN_GAIN:
        break
    selected_groups.append(cand); features_now += rf.ROSTER_GROUPS[cand]
    best_mae = trials[cand]; remaining.remove(cand)
    steps.append({"added": cand, "mae": best_mae})

display(pd.DataFrame(steps).round(3))
FEATURES = features_now
print("Selected roster groups:", selected_groups or "none -> roster adds nothing measurable on seasons < 2005")
if not selected_groups:
    FEATURES = BASE + ROSTER_COLS     # still evaluate the full roster set on the test seasons for transparency
    print("Evaluating with ALL roster features anyway.")

## 4. Evaluation on the test seasons (walk-forward)

In [ ]:
base_models = {
    "Ridge + roster": lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30))),
    "Gradient boosting + roster": lambda: HistGradientBoostingRegressor(max_depth=3, learning_rate=0.03, max_iter=250,
                                                                       min_samples_leaf=20, l2_regularization=1.0,
                                                                       random_state=RS),
}
models = dict(base_models)
models["Ensemble + roster"] = lambda: VotingRegressor([(n, f()) for n, f in base_models.items()])

preds = {
    "Naive: previous season": ru.baseline_scores(df, "pts_z_lag1", EVAL_YEARS),
    "Ridge (team features only)": ru.walk_forward(
        df, lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30))), BASE, EVAL_YEARS),
}
for name, factory in models.items():
    preds[name] = ru.walk_forward(df, factory, FEATURES, EVAL_YEARS)

ranked = {n: ru.rank_predictions(p, df) for n, p in preds.items()}
season_tables = {n: ru.season_metrics(r) for n, r in ranked.items()}
summary = pd.DataFrame({n: ru.summarise(t) for n, t in season_tables.items()}).T.sort_values("mae")
n_typical = int(round(ranked["Naive: previous season"]["conf_size"].mean()))
print(f"Random reference: MAE {ru.random_ranking_mae(n_typical):.2f}, top-{ru.PLAYOFF_SPOTS} overlap {ru.PLAYOFF_SPOTS/n_typical:.2f}")
display(summary.round(3))

fig, ax = plt.subplots(figsize=(9, 4.5))
for n, t in season_tables.items():
    s = t.groupby("year")["mae"].mean()
    ax.plot(s.index, s.values, marker="o", label=n, ls="--" if n.startswith("Naive") else "-")
ax.set_xlabel("Season"); ax.set_ylabel("MAE (rank positions)"); ax.set_title("Error per test season")
ax.legend(fontsize=8); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## 5. Is the gain real? (paired bootstrap over whole rankings)

In [ ]:
key = ["year", "tmID"]
ref_table = ranked["Naive: previous season"].set_index(key).sort_index()
idx = ref_table.index
gid = (ref_table["confID"] + "-" + idx.get_level_values("year").astype(str)).to_numpy()
abs_err = {n: r.set_index(key).sort_index()["abs_err"].reindex(idx).to_numpy() for n, r in ranked.items()}

rows = []
for name in ranked:
    row = {"model": name}
    for tag, ref in [("naive", "Naive: previous season"), ("team_only_ridge", "Ridge (team features only)")]:
        if name != ref:
            gain, (lo, hi) = ru.paired_bootstrap(abs_err[ref], abs_err[name], gid)
            row.update({f"gain_vs_{tag}": gain, f"ci_low_{tag}": lo, f"ci_high_{tag}": hi})
    rows.append(row)
significance = pd.DataFrame(rows).set_index("model").loc[summary.index]
display(significance.round(3))

## 6. Verdict

In [ ]:
roster_models = [n for n in summary.index if n.endswith("+ roster")]
best = summary.loc[roster_models].sort_values(["mae", "spearman"], ascending=[True, False]).index[0]
s, g = summary.loc[best], significance.loc[best]
ref = summary.loc["Ridge (team features only)"]
print(f"BEST ROSTER MODEL: {best}")
print(f"  MAE {s['mae']:.2f} | Spearman {s['spearman']:.2f} | top-8 overlap {s['top_overlap']:.0%}")
print(f"  team-only Ridge : MAE {ref['mae']:.2f} | Spearman {ref['spearman']:.2f} | top-8 overlap {ref['top_overlap']:.0%}")
print(f"  gain vs naive         : {g['gain_vs_naive']:+.2f} positions, 95% CI [{g['ci_low_naive']:+.2f}, {g['ci_high_naive']:+.2f}]")
print(f"  gain vs team-only Ridge: {g['gain_vs_team_only_ridge']:+.2f} positions, 95% CI [{g['ci_low_team_only_ridge']:+.2f}, {g['ci_high_team_only_ridge']:+.2f}]")
if g["ci_low_team_only_ridge"] > 0:
    print("-> Roster information brings a statistically reliable gain over team-only features.")
elif g["gain_vs_team_only_ridge"] > 0:
    print("-> Roster information helps slightly, but the gain is within noise on these seasons.")
else:
    print("-> Roster information does not help here.")
print("Reminder: the roster of the predicted season includes in-season arrivals (see assumption at the top).")

best_ranked = ranked[best]
last_year = EVAL_YEARS[-1]
for conf, grp in best_ranked[best_ranked["year"] == last_year].groupby("confID"):
    print(f"\nSeason {last_year} - conference {conf} - {best}")
    display(grp.sort_values("pred_rank")[["pred_rank", "tmID", "conf_rank", "err", "score"]].rename(
        columns={"conf_rank": "actual_rank"}).round(3))
best_ranked.to_csv("ranking_predictions_roster_output.csv", index=False)
print("\nSaved -> ranking_predictions_roster_output.csv")